<div>
<center><img src="../assets/Flux-logo.svg" width="360"/></center>
</div>

<div style="background:linear-gradient(90deg,#036291 0%,#91C2D8 100%);padding:20px 26px;border-radius:10px;border-left:10px solid #D9A441;margin-top:18px">
<h1 style="margin:0;color:#ffffff">Module 1: Flux Foundations &amp; Hierarchical Scheduling</h1>
<p style="margin:6px 0 0 0;color:#DCECF4;font-size:15px">Submitting work, batch jobs, and the hierarchy</p>
<p style="margin:2px 0 0 0;color:#DCECF4;font-size:13px">SC26 &middot; Chicago &middot; November 2026</p>
</div>

# Welcome

Flux is a resource manager and scheduler for HPC centers and clouds. Two things set it
apart from schedulers you may already know. Resources are a graph rather than a flat list
of nodes, and any Flux job can itself be a complete Flux instance with its own scheduler
and queue. That second property is hierarchical scheduling, and most of this module
builds toward it.

| Module | What you will do |
|---|---|
| **1. Foundations** | Submit jobs, run batch scripts, build a hierarchy, write a plugin |
| **2. Converged Environments** | Flux in the cloud, Usernetes, Kubeflow Trainer, quantum simulation |
| **3. Open Forum** | Lightning talks, trivia, and jokes of uneven quality |

Module 1 is three notebooks: this one, the [Python SDK](02_flux_python_sdk.ipynb), and
[instances, internals, and plugins](03_flux_instances_and_plugins.ipynb). Each ends with
optional side quests for anyone who finishes early.

## Set up your workspace

Click <button data-commandLinker-command="terminal:open" data-name="flux" href="#">this button</button>
to open a terminal, then drag it to sit beside this notebook. Every command here is meant
to be copied into that terminal. The notebook is the recipe card; the terminal is the kitchen.

```bash
cd /home/ubuntu/tutorial/module1
```

The source for all of this is in [flux-framework/Tutorials](https://github.com/flux-framework/Tutorials/tree/master/2026/SC26).
More Python examples live in `assets/flux-workflow-examples`, visible in the file browser on the left.

## Getting help

`flux help` lists the top-level commands. `flux help <command>` opens its manpage (press
`q` to leave), and every command also takes `--help`.

```bash
flux help
flux help jobs
flux submit --help
```

## What do I have to work with?

Flux discovers hardware with [hwloc](https://github.com/open-mpi/hwloc) and stores it in
a resource graph. On a multi-user system instance, administrators can hand it a
[static configuration](https://flux-framework.readthedocs.io/projects/flux-core/en/latest/guide/admin.html#configuration)
instead. Either way, `flux resource` is how you look at it.

```bash
flux resource list
```
```console
     STATE NNODES NCORES NGPUS NODELIST
      free      4    256     0 ip-10-0-25-[10,10,10,10]
 allocated      0      0     0
      down      0      0     0
```

Your numbers will differ. This tutorial runs under `flux start --test-size=4`, which
starts four Flux brokers on one machine, so the same host shows up four times as four
"nodes". `flux resource status` shows which nodes are up, and `flux queue list` shows the
queues and their limits.

```bash
flux resource status
flux queue list
```

## Flux commands, for Slurm users

If you know Slurm, most of Flux maps directly. A larger table covering LSF and Moab is
[here](https://hpc.llnl.gov/banks-jobs/running-jobs/batch-system-cross-reference-guides).

| Operation | Slurm | Flux |
|---|---|---|
| Run one job and wait for it | `srun` | `flux run` |
| Run one job interactively | `srun --pty` | `flux run -o pty.interactive` |
| Submit one job and return | | `flux submit` |
| Submit many jobs | | `flux bulksubmit`, `flux submit --cc` |
| Follow a job's output | | `flux watch` |
| Query jobs | `squeue`, `scontrol show job` | `flux jobs`, `flux job info` |
| Cancel a job | `scancel` | `flux cancel` |
| Interactive allocation | `salloc` | `flux alloc` |
| Batch script | `sbatch` | `flux batch` |

## flux run

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Run one job and wait for it</span>
</div>

`flux run` submits a job, attaches to it, streams its output to your terminal, and exits
with the job's exit code.

```bash
flux run sh -c 'sleep 5 && echo hello'
```

A non-zero exit is reported:

```bash
flux run /bin/false
flux-job: task(s) exited with exit code 1
```

Two quick `Ctrl-C`s cancel a running job. `Ctrl-C` then `Ctrl-Z` detaches and leaves it
running; `flux job attach JOBID` reattaches. The resource flags are `-N` nodes, `-n` tasks,
and `-c` cores per task:

```bash
flux run -N1 -n1 -c1 hostname
flux run -n4 --label-io --time-limit=5s hostname
```

Build the MPI hello program in this directory as a job, then run it with 32 tasks on one
node (adjust `-n` if you have fewer cores):

```bash
flux run make
flux run -N1 -n32 -c2 --exclusive ./hello
```

## flux submit

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Submit one job and return</span>
</div>

Same job, but `flux submit` prints the job ID and returns immediately.

```bash
flux submit hostname
```

`flux job last` returns the most recent job ID, and `flux job attach` connects to it,
blocking if it is still running:

```bash
flux job attach $(flux job last)
```

Redirect output to files with `--output` and `--error`:

```bash
flux submit --output=/tmp/deep-dish.txt --error=/tmp/deep-dish.txt echo "Order up for $(whoami)!"
cat /tmp/deep-dish.txt
```

Flags worth knowing. These are shared by `run`, `submit`, `batch`, and `alloc`:

| Flag | Meaning |
|---|---|
| `-N`, `-n`, `-c` | nodes, tasks, cores per task (`-c1` is the default) |
| `--job-name=NAME` | a name you will recognize in `flux jobs` |
| `--time-limit=5s` | kill the job after this long (`30m`, `2h`, ...) |
| `--requires=host:NAME` | pin a job to a rank or host |
| `--dependency=afterok:JOBID` | run only after another job succeeds |
| `--cc=IDSET` | submit copies of the same job |
| `--output`, `--error` | write output to files instead of the KVS |

## flux bulksubmit and --cc

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Submit many jobs</span>
</div>

`flux bulksubmit` substitutes inputs into a command line, like `xargs` or GNU `parallel`,
except each one becomes a Flux job with access to the whole instance:

```bash
flux bulksubmit --wait echo {} ::: sausage giardiniera mozzarella
```

`--cc=IDSET` submits carbon copies, with `{cc}` replaced by the index. `--watch` implies
`--wait` and streams output as it arrives; `--progress --jps` adds a progress bar and a
jobs-per-second rate:

```bash
flux submit --watch --cc=1-4 echo "Hello from copy {cc}"
flux submit --cc=1-100 --watch --progress --jps hostname
```

The eight scripts in `bulksubmit/` each print one step of a Chicago-style hot dog. Submit
them all at once, then attach to each in turn:

```bash
for jobid in $(flux submit --cc=0-7 bulksubmit/{cc}.sh); do flux job attach $jobid; done
```

Jobs of different shapes are co-scheduled without oversubscribing cores. These two run
forever on purpose; you will cancel them shortly.

```bash
flux submit --nodes=2 --ntasks=2 --job-name=deepdish sleep inf
flux submit --nodes=1 --ntasks=1 --job-name=extracheese sleep inf
```

## flux watch

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">👀 Follow a job's output</span>
</div>

`flux watch` is a read-only `flux job attach` that can follow several jobs at once:
every running job with `--active`, or every job you have ever run with `--all`.
[job-watch.sh](job-watch.sh) prints a poem one line at a time so there is something to see:

```bash
flux submit ./job-watch.sh
flux watch $(flux job last)
```

## flux jobs

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Query jobs</span>
</div>

```bash
flux submit sleep inf
flux jobs
```

In a real terminal, instances are blue, failed jobs red, and completed jobs green. `-a`
includes inactive jobs. `--format` picks fields, and `--no-header` makes the output easy
to use from a script:

```bash
flux jobs -a
flux jobs -a --no-header --format="{id.f58} {name:>12} {status}"
flux jobs --no-header --format="{status}" $(flux job last)
```

## flux cancel

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Cancel jobs</span>
</div>

Several of the jobs above will never exit on their own. Cancel everything:

```bash
flux cancel --all
flux jobs
```

## flux alloc

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Interactive allocation</span>
</div>

`flux alloc` requests resources and drops you into a shell inside a new Flux instance that
owns them. The new instance has its own scheduler and its own queue.

```bash
flux resource list      # what the parent has
flux alloc -N2          # a child instance on two of its "nodes"
flux resource list      # what the child has
flux uptime             # depth 1: one layer down
exit
```

## flux batch

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Batch script</span>
</div>

`flux batch` does the same thing without the shell. It starts a nested instance across
the allocated resources and runs your script on rank 0 of that instance. The script
submits work to the child, and nothing stops it from calling `flux batch` again to nest
further.

Look at [sleep_batch.sh](sleep_batch.sh). Its `#FLUX:` directives stand in for
command-line flags. Submit it twice and watch two instances come up side by side:

```bash
flux batch ./sleep_batch.sh
flux batch ./sleep_batch.sh
flux jobs
```

`flux job info` returns pieces of a job's record; `R` is the resource set it was given.
Attaching to a batch job points you at its output file:

```bash
flux job info $(flux job last) R
flux job attach $(flux job last)
```

### Submitting from inside a batch script

[hello-batch.sh](hello-batch.sh) submits four jobs to its own instance with
`--flags=waitable`, then `flux job wait --all` blocks until they finish. (`flux queue drain`
is the alternative when you did not flag the jobs.) The batch job itself is submitted as
waitable too, so you can wait for it from out here:

```bash
flux batch --flags=waitable --output=/tmp/flux-batch.out -N1 ./hello-batch.sh
flux job wait
cat /tmp/hello-batch-*.out
```
```console
Hello job 1 from ip-10-0-25-10 💛️
Hello job 2 from ip-10-0-25-10 💚️
Hello job 3 from ip-10-0-25-10 💙️
Hello job 4 from ip-10-0-25-10 💜️
```

## The hierarchy 🍕

> A deep dish is built in layers, and each layer does its own job. A single-queue
> scheduler is a flat pizza: one surface, and throughput bounded by how fast that surface
> works. `flux uptime` tells you which layer you are standing on. `depth 0` is the pan.

Every `flux alloc` and `flux batch` you just ran created a complete Flux instance, with
its own scheduler, inside a job. That is the whole trick. A central scheduler that
handles one job per second (left) handles three per second the moment you put three
instances under it (right), because each child schedules independently and the parent
only ever sees three jobs.

<table>
<tr>
    <td>
<img src="../img/single-submit.png" style="float:left; margin-top:30px" width="350px">
    </td>
    <td>
<img src="../img/instance-submit.png" style="float:right; margin-top:-20px" width="550px">
    </td>
    </tr>
</table>

It compounds. In the experiment in our
[learning guide](https://flux-framework.readthedocs.io/en/latest/guides/learning_guide.html#fully-hierarchical-resource-management-techniques)
(Figure 10), three levels reached about 500 jobs per second against roughly one for a
single level. Each child can also run a different scheduling policy, so you can tune a
sub-tree for your workload without touching the system instance.

[sub_job1.sh](sub_job1.sh) is a batch script that itself calls `flux batch` on
[sub_job2.sh](sub_job2.sh), which runs a `sleep`. Submit it, then look at the tree of
everything you have run:

```bash
flux batch ./sub_job1.sh
flux pstree -a
```
```console
.
├── ./sub_job1.sh
│   └── ./sub_job2.sh
│       └── sleep:R
├── ./hello-batch.sh:CD
├── job-watch.sh:CD
├── 7.sh:CD
├── 6.sh:CD
...
```

`flux pstree -a -X` adds detail. Deeper trees are in the
[docs](https://flux-framework.readthedocs.io/en/latest/jobs/hierarchies.html#flux-pstree-command).

## flux start

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Start an instance</span>
</div>

Everything in this tutorial runs under one command:

```bash
flux start --test-size=4
```

`flux start` launches a Flux instance. It can be the system resource manager, a job
inside Slurm or LSF or another Flux, or, as here, a test instance of four brokers on one
machine. Without a command it gives you a shell; with one, it runs the command and exits:

```bash
flux start --test-size=4 flux getattr size
4
```

## Four ways to do work

|  | creates a sub-instance | runs a program |
|---|---|---|
| interactive | `flux alloc` | `flux run` |
| in the background | `flux batch` | `flux submit` |

Submit as much as you like inside an `alloc` or `batch` instance. The parent only ever
sees one job.

<br>

## 🧭 Side quests

<div class="alert alert-block" style="background-color:#D9A441;color:#06293D">
<span style="font-weight:600">Finished early?</span> Pick one. Each takes about ten minutes, and an answer is folded up underneath it. Try before you unfold.
</div>

### 1. Build the hot dog in order

The eight scripts in `bulksubmit/` are the steps of a Chicago-style hot dog, and the
kitchen is strict about order. `flux submit --cc=0-7` runs them in parallel, so
`flux watch --active` shows the steps scrambled. Make them run strictly one after another,
using only `flux submit`, and confirm with a single `flux watch --active`.

Hints: `--dependency=afterok:JOBID`. `flux submit` prints the job ID. A `for` loop.

<details><summary>One answer</summary>

```bash
prev=$(flux submit bulksubmit/0.sh)
for i in 1 2 3 4 5 6 7; do
  prev=$(flux submit --dependency=afterok:$prev bulksubmit/$i.sh)
done
flux watch --active
```

While they run, `flux jobs -o "{id.f58} {name} {status} {dependencies}"` shows seven jobs
waiting in `DEPEND`. Follow-up: cancel step 3 mid-chain. What happens to steps 4 through 8,
and how does `--dependency=after:` (no `ok`) change that?
</details>

### 2. The six-core oven

| Pizza | Cores | Bake time |
|---|---|---|
| margherita | 1 | 20s |
| pepperoni | 1 | 20s |
| deep dish | 4 | 30s |
| sicilian | 4 | 30s |
| calzone | 2 | 10s |
| stromboli | 2 | 10s |

Get an oven of exactly six cores with `flux alloc -n6`. Inside it, submit all six pizzas
as jobs (`--job-name` for the pizza, `sleep` for the bake, `-c` for the cores) and measure
the makespan: first start to last finish.

```bash
flux alloc -n6
flux submit -c1 --job-name=margherita sleep 20
# ... the other five ...
flux watch --active
flux jobs -a -o "{name:>10} {t_run!d:%M:%S} {t_inactive!d:%M:%S}"
exit
```

The pizzas need 320 core-seconds in total, so six cores could in principle finish in 54
seconds. Can you get there? If not, why not? Does submission order matter? What changes
in an eight-core oven?

<details><summary>Answer</summary>

54 seconds is unreachable. The two 4-core pizzas cannot both fit in a six-core oven, so
one cannot start until the other finishes at 30s, and the makespan is at least 60s. 60 is
achievable: deep dish with margherita and pepperoni first, then sicilian with calzone and
stromboli. Jobs are not liquid. A core-seconds floor counts area, not shape.

With eight cores the area floor is 40s but the real minimum is 50s, and now order
matters. Under a strict first-come-first-served policy (Fluxion's default), a 4-core
pizza waiting at the head of the queue blocks everything behind it, so submitting the
2-core pizzas first can cost you 70s.
</details>

### 3. Who burned the pizza?

A job died. Find out why, using only `flux jobs` and `flux job eventlog`:

```bash
flux submit --job-name=burnt --time-limit=3s sleep 30
```

Wait a few seconds, then investigate. What status is the job in, which event killed it,
and what does the `-H` flag to `flux job eventlog` do?

<details><summary>Answer</summary>

```bash
flux jobs -a
flux job eventlog -H $(flux job last)
```

The status is `TIMEOUT`, and the eventlog has an `exception type="timeout"` entry with
the note `resource allocation expired`. `-H` prints each event's time relative to the
first event instead of as an absolute timestamp. `flux jobs -f timeout` finds every job
that died this way, and `flux job eventlog -p exec JOBID` shows the job shell's side of
the same story.
</details>

<div style="background:#DCECF4;border-left:6px solid #D9A441;padding:12px 18px;color:#06293D"><strong>Module 1, Notebook 1 complete</strong></div>

You submitted work four ways, read the queue, built a hierarchy, and read it back with
`flux pstree`.

Next: the [Python SDK](02_flux_python_sdk.ipynb), the one notebook where you run cells
directly instead of copying to a terminal.